# 10 - Record selection of Shalabi et al.

Shalabi et al. (2026) train on the three-component accelerograms (HNE, HNN, HNZ channels) of three ESM archives:
ITACA (Italy), HEAD (Greece) and TR-NSMN (Turkey), with magnitude of at least 4; they then draw 22,250 records per
class. `training.record_selection` in `configs/default.yaml` reproduces such a selection on ESM25 (networks, channel
code, orientations, minimum magnitude), and the experiments (notebooks 05, 08a, 08) use only the selected records.

This notebook:

1. describes the network, channel and orientation codes of the records;
2. counts the records kept after each criterion of the configured selection, against the 22,250 records per class of
   Shalabi et al.;
3. evaluates the replication run trained on all the records (`RUN_NAME`, record split) on the selected validation
   records: a preview of the selection without training. A run trained on the selection is evaluated on the same
   records, because the selection is applied to the same split.

The records are those of the CV image file of notebook 02d (records without data excluded), the same for every
image setting.

## 0. Setup

In [ ]:
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd

from wqu.config import CV_GOOD_SOURCE, load_config
from wqu.data.selection import (
    CHANNEL_COLUMN,
    NETWORK_COLUMN,
    ORIENTATION_COLUMNS,
    class_counts,
    selection_mask,
    selection_steps,
    step_counts,
)
from wqu.evaluation.padding import metrics_by_group
from wqu.preprocessing.features import representation_feature_path
from wqu.preprocessing.pipeline import read_feature_records
from wqu.training.metrics import subset_metrics
from wqu.training.trainer import VALIDATION_PREDICTIONS_FILE, find_run_dir

CONFIG_PATH = Path("..") / "configs" / "default.yaml"
# Replication run trained on all the records (CV images, record split).
RUN_NAME = "obspy_spectrogram_20261124_190536"
PREDICTION_COLUMN = "probability_deterministic"
# Records per class drawn by Shalabi et al. (max_samples_per_class in their dataset.py).
SHALABI_RECORDS_PER_CLASS = 22_250
INSTRUMENT_COLUMN = "station_instrument_type_code"
# Rows shown in the tables of codes, and records of each class below which a group gets no metrics.
MAX_ROWS = 30
MIN_RECORDS_PER_CLASS = 30

config = load_config(CONFIG_PATH)
label_column = config.columns.label
n_bins = config.training.calibration_bins
selection = config.training.record_selection
class_names = {subset.label: subset.name for subset in config.data.subsets}

cv_settings = replace(config.preprocessing.obspy_spectrogram, good_source=CV_GOOD_SOURCE)
features_path = representation_feature_path(replace(config.preprocessing, obspy_spectrogram=cv_settings),
                                            config.training.feature_source, config.training.representation)
records = read_feature_records(features_path, config.columns.text)
labels = records[label_column]
print(f"Records: {len(records):,} ({features_path.parent.name}); classes: {class_names}")
print(f"Selection: {'none' if selection.is_empty else selection}")

## 1. Networks, channels and orientations

Number of records per code and class. The channel code is the band and instrument code of the three components (HN:
high sample rate accelerometer); the orientation codes are those of the u, v and w components (E, N, Z or numbered).

In [ ]:
code_columns = [NETWORK_COLUMN, CHANNEL_COLUMN, INSTRUMENT_COLUMN, *ORIENTATION_COLUMNS]
display(pd.DataFrame({
    "distinct values": records[code_columns].nunique(),
    "missing": records[code_columns].isna().sum(),
    "examples": [", ".join(map(str, records[column].dropna().unique()[:6])) for column in code_columns],
}))
for column in code_columns:
    counts = class_counts(labels, records[column], class_names)
    print(f"\n{column} ({len(counts)} values, the {min(MAX_ROWS, len(counts))} largest shown)")
    display(counts.head(MAX_ROWS).round(3))

## 2. Records kept by the selection

Records per class after each criterion, applied in the order network, channel, orientation, magnitude (each row keeps
the records that meet that criterion and all the previous ones). Shalabi et al. draw 22,250 records per class: a
selection that leaves fewer records of a class cannot be the one of their dataset, at least on ESM25.

In [ ]:
steps = selection_steps(records, selection)
counts = step_counts(labels, steps, class_names)
display(counts.round(3))

selected = selection_mask(records, selection)
final = counts.iloc[-1]
for class_name in class_names.values():
    kept = int(final[class_name])
    print(f"{class_name}: {kept:,} records kept, {kept / SHALABI_RECORDS_PER_CLASS:.2f} times the "
          f"{SHALABI_RECORDS_PER_CLASS:,} of Shalabi et al.")
print("\nSelected records per network")
display(class_counts(labels[selected], records.loc[selected, NETWORK_COLUMN], class_names).round(3))

## 3. Preview on the validation records of the replication run

Metrics of the run trained on all the records on the validation records kept after each step, then per network and
per channel code (groups with at least `MIN_RECORDS_PER_CLASS` records of each class). AUC and balanced accuracy do
not depend on the fraction of bad records, so the rows can be compared even if it differs. The record-weighted mean
of the AUC within the networks, against the AUC of all the records, tells how much the network relies on the identity
of the seismic network (whose fraction of bad records varies widely): a much lower mean would mean that part of the
separation comes from the network.

In [ ]:
run_dir = find_run_dir(config.training.runs_dir, config.training.representation, RUN_NAME)
predictions = pd.read_csv(run_dir / VALIDATION_PREDICTIONS_FILE)
validation_records = records.loc[predictions["record"].to_numpy()].reset_index(drop=True)
probabilities = predictions[PREDICTION_COLUMN].to_numpy()
corrected = predictions[f"{PREDICTION_COLUMN}_corrected"].to_numpy()
validation_labels = predictions[label_column].to_numpy()
print(f"Run: {run_dir.name}; validation records: {len(predictions):,}")

validation_steps = selection_steps(validation_records, selection)
display(pd.DataFrame({
    name: subset_metrics(probabilities, corrected, validation_labels, mask.to_numpy(), n_bins)
    for name, mask in validation_steps.items()
}).T.round(4))


def group_table(groups: pd.Series, probabilities: np.ndarray, corrected: np.ndarray, labels: np.ndarray,
                n_bins: int, min_per_class: int) -> pd.DataFrame:
    """Metrics per group, for the groups with at least ``min_per_class`` records of each class.

    Args:
        groups: Group of every record, aligned with the probabilities.
        probabilities: Raw probabilities of the bad class.
        corrected: Prior-corrected probabilities of the bad class.
        labels: Labels (1 = bad).
        n_bins: Bins of the calibration error.
        min_per_class: Minimum number of good and of bad records of a group.

    Returns:
        Output of ``metrics_by_group`` restricted to those groups, sorted by decreasing number of records.
    """
    table = metrics_by_group(probabilities, corrected, labels, groups.astype("category"), n_bins)
    n_bad = (table["records"] * table["bad_rate"]).round()
    enough = np.minimum(n_bad, table["records"] - n_bad) >= min_per_class
    return table[enough].sort_values("records", ascending=False)


group_arguments = (probabilities, corrected, validation_labels, n_bins, MIN_RECORDS_PER_CLASS)
networks = group_table(validation_records[NETWORK_COLUMN], *group_arguments)
within_auc = np.average(networks["auc"], weights=networks["records"])
print(f"\nPer network ({len(networks)} networks); record-weighted mean of the AUC within the networks: {within_auc:.4f}")
display(networks.round(4))
print("\nPer channel code")
display(group_table(validation_records[CHANNEL_COLUMN], *group_arguments).round(4))